# Analytic verification of the asymptotic cascade analysis

Symbolic (machine-checked) verification of the identities and scaling laws behind the
threshold-/degree-/marginal-dominated regimes of $\kappa(z)$.

Each cell prints `True` when the symbolic identity holds.

In [1]:
import sympy as sp
sp.init_printing()
n, p = sp.symbols('n p', positive=True)
k = sp.symbols('k', positive=True, integer=True)
a, b = sp.symbols('a b', positive=True)
print('sympy', sp.__version__)

sympy 1.14.0


## 1. Beta-binomial pmf identity

$P(K{=}k)=\binom{n}{k}\frac{B(k+a,\,n-k+b)}{B(a,b)}$ is the Beta-mixture of Binomial.
The only nontrivial step is the Beta kernel $\int_0^1 p^{A-1}(1-p)^{B-1}dp=B(A,B)$.

In [2]:
np_, kp = sp.symbols('np kp', positive=True)
kernel = sp.integrate(p**(kp+a-1)*(1-p)**(np_-kp+b-1), (p,0,1), conds='none')
print('mixture kernel == B(k+a, n-k+b):',
      sp.simplify(kernel - sp.beta(kp+a, np_-kp+b)) == 0)

mixture kernel == B(k+a, n-k+b): True


## 2. Mixture moments

$\mathbb E[p^{q}]=\int_0^1 \frac{p^{q+a-1}(1-p)^{b-1}}{B(a,b)}dp=\frac{B(a+q,b)}{B(a,b)}$.

In [3]:
q = sp.symbols('q', positive=True)
mom = sp.integrate(p**(q+a-1)*(1-p)**(b-1), (p,0,1), conds='none')/sp.beta(a,b)
print('E[p^q] == B(a+q,b)/B(a,b):',
      sp.simplify(mom - sp.beta(a+q,b)/sp.beta(a,b)) == 0)

E[p^q] == B(a+q,b)/B(a,b): True


## 3. Stirling / Gamma-ratio limits

$\frac{n!}{(n-k)!}\to n^{k}$ and $\frac{\Gamma(n+c)}{\Gamma(n+d)}\to n^{c-d}$ (fixed $k$, $n\to\infty$).
Concrete $a=\tfrac34,\,b=3$; symbolic positive integer $k$.

In [4]:
a0, b0 = sp.Rational(3,4), sp.Integer(3)
L1 = sp.limit(sp.factorial(n)/sp.factorial(n-k)/n**k, n, sp.oo)
L2 = sp.limit(sp.gamma(n-k+b0)/(sp.gamma(n+a0+b0)*n**(-(k+a0))), n, sp.oo)
L3 = sp.limit(sp.gamma(n+1)/(sp.gamma(n+sp.Rational(15,4))*n**(1-sp.Rational(15,4))), n, sp.oo)
print('n!/(n-k)! ~ n^k                       :', L1)
print('Gamma(n-k+b)/Gamma(n+a+b) ~ n^-(k+a)  :', L2)
print('Gamma(n+c)/Gamma(n+d) ~ n^(c-d)       :', L3)

n!/(n-k)! ~ n^k                       : 1
Gamma(n-k+b)/Gamma(n+a+b) ~ n^-(k+a)  : 1
Gamma(n+c)/Gamma(n+d) ~ n^(c-d)       : 1


## 4. Fixed-$k$ large-$n$ pmf

Combining 1 & 3:
$P(k;n)\sim n^{-a}\dfrac{\Gamma(k+a)\,\Gamma(a+b)}{\Gamma(a)\,\Gamma(b)\,\Gamma(k+1)}$.

In [5]:
P = (sp.gamma(n+1)/(sp.gamma(k+1)*sp.gamma(n-k+1))
     * sp.gamma(k+a0)*sp.gamma(n-k+b0)/sp.gamma(n+a0+b0)/sp.beta(a0,b0))
lhs = sp.limit(n**a0 * P, n, sp.oo)
rhs = sp.gamma(k+a0)*sp.gamma(a0+b0)/(sp.gamma(k+1)*sp.gamma(a0)*sp.gamma(b0))
print('fixed-k large-n asymptotics:', sp.simplify(lhs-rhs) == 0)

fixed-k large-n asymptotics: True


## 5. Threshold-dominated regime ($\alpha_{th}<a+2$)

For $k\gg1$, $F(1/k)\sim C_{th}k^{-\alpha_{th}}$, so
$\sum_k k(k{-}1)P(k)F(1/k)\sim C_{th}\,\mathbb E[K^{2-\alpha_{th}}]\sim C_{th}n^{2-\alpha_{th}}\frac{B(a+2-\alpha_{th},b)}{B(a,b)}$,
using $\mathbb E[K^{s}]\sim n^{s}\mathbb E[p^{s}]$ for $s>0$ with the moment finite iff $\alpha_{th}<a+2$. With $z=nm_d$ this gives $\kappa(z)\sim C_1 z^{1-\alpha_{th}}$, $C_1=C_{th}\,m_d^{\alpha_{th}-2}\frac{B(a+2-\alpha_{th},b)}{B(a,b)}$.

In [6]:
th, z = sp.symbols('th z', positive=True)
md = a/(a+b)
from_sum = (1/(n*md))*th*n**(2-th)*sp.beta(a+2-th,b)/sp.beta(a,b)
in_z     = th*md**(th-2)*z**(1-th)*sp.beta(a+2-th,b)/sp.beta(a,b)
print('n=z/m_d bookkeeping -> C1 z^(1-alpha_th):',
      sp.simplify(from_sum.subs(n, z/md) - in_z) == 0)
print('finite moment iff alpha_th < a+2  (kernel p^(a+1-alpha_th) integrable):',
      sp.simplify(sp.beta(a+2-th,b)) != sp.oo)

n=z/m_d bookkeeping -> C1 z^(1-alpha_th): True
finite moment iff alpha_th < a+2  (kernel p^(a+1-alpha_th) integrable): True


## 6. Degree-dominated / marginal regimes

Summand $\sim k^{1+a-\alpha_{th}}$. Converges iff $\alpha_{th}>a+2$ (exponent $<-1$); at $\alpha_{th}=a+2$ it is $\sum 1/k\sim\ln n$, giving the $\ln z$ correction $\kappa\sim C_c\,z^{-(a+1)}\ln n$.

In [7]:
kk = sp.Symbol('k', positive=True, integer=True)
print('degree-dominated  sum k^-3/2  (a=3/4, alpha_th=3.25):',
      sp.summation(kk**sp.Rational(-3,2), (kk,2,sp.oo)))
print('marginal  sum 1/k  diverges -> log :', sp.summation(1/kk, (kk,2,sp.oo)))
print('H_n - ln n -> EulerGamma          :', sp.limit(sp.harmonic(n)-sp.log(n), n, sp.oo))

degree-dominated  sum k^-3/2  (a=3/4, alpha_th=3.25): zeta(3/2, 2)
marginal  sum 1/k  diverges -> log : oo
H_n - ln n -> EulerGamma          : EulerGamma


## 7. Independent numeric cross-check (asserts)

In [8]:
import numpy as np
from scipy.special import beta as B, gammaln, loggamma
a0n, b0n = 0.75, 3.0
nnum = 4000
kn = np.arange(0, nnum+1)
logp = (gammaln(nnum+1)-gammaln(kn+1)-gammaln(nnum-kn+1)
        +loggamma(kn+a0n)+loggamma(nnum-kn+b0n)+loggamma(a0n+b0n)
        -(loggamma(nnum+a0n+b0n)+loggamma(a0n)+loggamma(b0n)))
pk = np.exp(logp)
m = a0n/(a0n+b0n)
assert abs(pk.sum()-1) < 1e-9                                  # normalisation
assert abs((kn*pk).sum() - nnum*m) < 1e-6                      # E[K] = n m_d
assert abs((pk*kn*(kn-1)).sum() - nnum*(nnum-1)*B(a0n+2,b0n)/B(a0n,b0n)) < 1e-4  # E[K(K-1)]
print('all numeric asserts passed')

all numeric asserts passed


# Appendix A — line-by-line audit

Each numbered item corresponds to an equation in Appendix A of `main.tex` (and the supporting body equations). `ok(...)` prints PASS/FAIL.

In [9]:
from scipy.special import beta as Bf, betainc, gammaln, loggamma, zeta
from scipy.stats import poisson as po, betabinom, norm
import numpy as np
def ok(name, cond):
    print(('PASS ' if cond else 'FAIL ') + name)

## A.1 Threshold-distribution asymptotics (l.1244–1332)

In [10]:
import sympy as sp
t,u = sp.symbols('t u', positive=True)
A,B = sp.Rational(3,2), sp.Integer(3)      # concrete a,b for the bracket
integ = sp.integrate(t**(A-1)*sp.series((1-t)**(B-1),t,0,3).removeO(),(t,0,u))
Iu = sp.simplify(integ/sp.beta(A,B))
bracket = sp.simplify(Iu*sp.beta(A,B)*A/u**A)          # = 1 + a(1-b)/(a+1) u + O(u^2)
c = sp.simplify(sp.series(bracket-1,u,0,2).coeff(u,1))
ok('A.1 (l.1295) I_u coeff = a(1-b)/(a+1)', sp.simplify(c-A*(1-B)/(A+1))==0)

for b in (0.7,3.0):
    for uu in (0.05,0.2,0.5):
        assert abs(betainc(1,b,uu)-(1-(1-uu)**b))<1e-12
ok('A.1 (l.1385) I_u(1,b)=1-(1-u)^b', True)

al,bth,phibar,kk = sp.symbols('al bth phibar kk', positive=True)
m = al/(al+bth); Cth = m**al/(al*sp.beta(al,bth)*phibar**al)
Flead = (m/(kk*phibar))**al/(al*sp.beta(al,bth))
ok('A.1 (l.1311) F(1/k)=C_th k^-a', sp.simplify(Flead-Cth*kk**(-al))==0)

PASS A.1 (l.1295) I_u coeff = a(1-b)/(a+1)
PASS A.1 (l.1385) I_u(1,b)=1-(1-u)^b
PASS A.1 (l.1311) F(1/k)=C_th k^-a


## A.2 Erdős–Rényi cascade parameter (l.1335–1441)

In [11]:
k = sp.symbols('k', positive=True, integer=True)
ok('A.2 (l.1347) k(k-1)/k! = 1/(k-2)!', sp.simplify(k*(k-1)/sp.factorial(k)-1/sp.factorial(k-2))==0)
zz=7.3; K=np.arange(0,200); pk=po.pmf(K,zz)
ok('A.2 (l.654) sum_{k>=2}(k-1)p_k = z-1+p0', abs(((K[2:]-1)*pk[2:]).sum()-(zz-1+pk[0]))<1e-9)

def er_kappa(z,a,b,phi):
    K=np.arange(2,int(z+12*np.sqrt(z)+80)); pk=po.pmf(K,z)
    uu=(a/(a+b))/(K*phi); F=np.where(uu>=1,1.0,betainc(a,b,np.minimum(uu,1)))
    return np.sum(K*(K-1)*pk*F)/z
for (a,b,phi) in [(0.5,3.0,0.2),(1.5,3.0,0.2)]:
    zs=[400,1600,6400]; ks=[er_kappa(zz,a,b,phi) for zz in zs]
    slope=(np.log(ks[-1])-np.log(ks[0]))/(np.log(zs[-1])-np.log(zs[0]))
    ok(f'A.2 (l.1367) ER slope a={a}: {slope:.3f} -> {1-a}', abs(slope-(1-a))<0.05)

kinf=3.0/((1+3.0)*0.25)
ok(f'A.2 (l.1416) kappa_inf num={er_kappa(2e5,1.0,3.0,0.25):.5f} vs {kinf:.5f}',
   abs(er_kappa(2e5,1.0,3.0,0.25)-kinf)<1e-3)

PASS A.2 (l.1347) k(k-1)/k! = 1/(k-2)!
PASS A.2 (l.654) sum_{k>=2}(k-1)p_k = z-1+p0
PASS A.2 (l.1367) ER slope a=0.5: 0.501 -> 0.5
PASS A.2 (l.1367) ER slope a=1.5: -0.497 -> -0.5
PASS A.2 (l.1416) kappa_inf num=2.99997 vs 3.00000


## A.3 Beta-binomial distribution (l.1443–1501; body l.772–779)

In [12]:
ad,bd,n=0.75,3.0,400; K=np.arange(0,n+1); Pk=betabinom.pmf(K,n,ad,bd)
mn=ad/(ad+bd); var=n*ad*bd/(ad+bd)**2*(ad+bd+n)/(ad+bd+1)
ok('A.3 (l.774) BB mean = n a_d/(a_d+b_d)', abs((K*Pk).sum()-n*mn)<1e-6)
ok('A.3 (l.776) BB variance', abs((K**2*Pk).sum()-(n*mn)**2-var)<1e-3)
from scipy.stats import binom as _binom
n2=40000; p_fix=0.3
ok('A.3 (l.1492) fixed p: Var(K|p)/n^2 = p(1-p)/n -> 0', _binom.var(n2,p_fix)/n2**2 < 1e-4)

np_,kp = sp.symbols('np kp', positive=True)
kern = sp.integrate(p**(kp+ad-1)*(1-p)**(np_-kp+bd-1),(p,0,1),conds='none')
ok('A.3 (l.1459) mixture kernel = B(k+a_d,n-k+b_d)',
   sp.simplify(kern-sp.beta(kp+ad,np_-kp+bd))==0)

PASS A.3 (l.774) BB mean = n a_d/(a_d+b_d)
PASS A.3 (l.776) BB variance
PASS A.3 (l.1492) fixed p: Var(K|p)/n^2 = p(1-p)/n -> 0


PASS A.3 (l.1459) mixture kernel = B(k+a_d,n-k+b_d)


## A.4/A.5 The three regimes (l.1504–1738)

In [13]:
a,b = sp.symbols('a b', positive=True)
q = sp.symbols('q', positive=True)
mom = sp.integrate(p**(q+a-1)*(1-p)**(b-1),(p,0,1),conds='none')/sp.beta(a,b)
ok('A.4 (l.1567) E[p^q]=B(a+q,b)/B(a,b)', sp.simplify(mom-sp.beta(a+q,b)/sp.beta(a,b))==0)

def bb_kappa(n,ad,bd,ath,bth,phi):
    kk=np.arange(2,n+1)
    lp=(gammaln(n+1)-gammaln(kk+1)-gammaln(n-kk+1)+loggamma(kk+ad)+loggamma(n-kk+bd)+loggamma(ad+bd)
        -(loggamma(n+ad+bd)+loggamma(ad)+loggamma(bd)))
    pk=np.exp(lp); mth=ath/(ath+bth); uu=mth/(kk*phi)
    F=np.where(uu>=1,1.0,betainc(ath,bth,np.minimum(uu,1)))
    return np.sum(kk*(kk-1)*pk*F)/(n*ad/(ad+bd))

ad_,bd_,phi=0.75,3.0,0.2; md=ad_/(ad_+bd_)
ath=1.5; mth=ath/(ath+bd_); Cth=mth**ath/(ath*Bf(ath,bd_)*phi**ath)
C1=Cth*md**(ath-2)*Bf(ad_+2-ath,bd_)/Bf(ad_,bd_)
k=bb_kappa(20000,ad_,bd_,ath,bd_,phi); z=20000*md
ok(f'A.4 (l.1603) C1: kappa/asym={k/(C1*z**(1-ath)):.4f}', abs(k/(C1*z**(1-ath))-1)<0.01)

ath1=1.0; mth1=ath1/(ath1+bd_); Cth1=mth1**ath1/(ath1*Bf(ath1,bd_)*phi**ath1)
C1b=Cth1*md**(ath1-2)*Bf(ad_+2-ath1,bd_)/Bf(ad_,bd_); kinf=bd_/((1+bd_)*phi)
ok(f'A.4 (l.1416/l.1603) a_th=1: C1={C1b:.6f} == kappa_inf={kinf:.6f}', abs(C1b-kinf)<1e-12)

ad_s,aths = sp.symbols('ad_s aths', positive=True)
ok('A.5 (l.1681) degree summand exp = 1+a_d-a_th',
   sp.simplify((2+(ad_s-1)-aths)-(1+ad_s-aths))==0)
kk2=sp.Symbol('k', positive=True, integer=True)
ok('A.5 (l.1709) marginal summand ~ 1/k diverges -> ln', sp.summation(1/kk2,(kk2,2,sp.oo))==sp.oo)
hh=sp.Symbol('hh', positive=True, integer=True)
ok('A.5 (l.1711) H_n - ln n -> gamma', sp.limit(sp.harmonic(hh)-sp.log(hh),hh,sp.oo)==sp.EulerGamma)
zln=sp.symbols('z', positive=True); adp=sp.symbols('adp', positive=True)
slope=sp.diff(-(adp+1)*sp.log(zln)+sp.log(sp.log(zln)), zln)*zln
ok('A.5 (l.1052) slope = -(a_d+1)+1/ln z', sp.simplify(slope-(-(adp+1)+1/sp.log(zln)))==0)

PASS A.4 (l.1567) E[p^q]=B(a+q,b)/B(a,b)
PASS A.4 (l.1603) C1: kappa/asym=0.9983
PASS A.4 (l.1416/l.1603) a_th=1: C1=3.750000 == kappa_inf=3.750000
PASS A.5 (l.1681) degree summand exp = 1+a_d-a_th
PASS A.5 (l.1709) marginal summand ~ 1/k diverges -> ln
PASS A.5 (l.1711) H_n - ln n -> gamma
PASS A.5 (l.1052) slope = -(a_d+1)+1/ln z


## Body cross-checks

In [14]:
g=np.linspace(3.0,4.0,400001); f=(zeta(g-2)-zeta(g-1))/zeta(g-1)
i=np.where(np.diff(np.sign(f-1)))[0]
ok(f'body (l.461) zeta gamma_d*={g[i[0]]:.4f} (paper 3.479)', abs(g[i[0]]-3.479)<0.01)

gam,ph=2.5,0.2; xmin=ph*(gam-2)/(gam-1)
ok('body (l.374) Pareto x_min => mean phi', abs(xmin*(gam-1)/(gam-2)-ph)<1e-12)

mu,sig,aa,bb_=0.4,0.1,0.0,1.0; al_=(aa-mu)/sig; be_=(bb_-mu)/sig
mtr=mu+sig*(norm.pdf(al_)-norm.pdf(be_))/(norm.cdf(be_)-norm.cdf(al_))
xs=np.linspace(0,1,400001); w=norm.pdf((xs-mu)/sig); w/=w.sum()
ok('body (l.417) truncated-Gaussian mean formula', abs((xs*w).sum()-mtr)<1e-4)

PASS body (l.461) zeta gamma_d*=3.4788 (paper 3.479)
PASS body (l.374) Pareto x_min => mean phi
PASS body (l.417) truncated-Gaussian mean formula


## Notes (not errors)

1. Marginal regime (l.1706→1717): derivation gives $\ln n$, text writes $\ln z$; asymptotically equivalent (differ by a term subleading to $z^{-(\alpha_d+1)}\ln z$). Figure uses $\ln(z/m_d)$.
2. Threshold-dominated moment substitution (l.1555) is a standard LLN step (validated numerically), not a proved uniform asymptotic.
3. Degree-dominated (l.1617) implicitly assumes uniform control of the $k\to n$ tail.